# Saz vokoderi v1 · eğitim · tek hücre

**Yapacağın üç şey:**
1. Üst menü: **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU → Kaydet**.
2. Aşağıdaki hücrenin solundaki **▶** düğmesine bas.
3. Açılan pencerede Google hesabını seçip **Drive'a izin ver**. **veri:** satırlarını görünce kalkabilirsin.

Gerisi kendiliğinden: veri, eğitim, ölçüm. Eğitim uzun (A100'de tahminen 8-9 saat); oturum koparsa ya da kota biterse
`DEVAM = 'son'` yapıp yeniden ▶: eğitim son kayıt noktasından (en çok 10 dk geriden) sürer.

- Sonuç Drive'da: `ses_nota/ses-modeli/runs/<tarih>-<AD>/` (en_iyi.pt, dogrulama.jsonl, ornek/, olcu/).

**İÇ TEST.** Veri ve model yalnız Drive'da kalır; paylaşılmaz.


In [ ]:
# Saz vokoderi v1 -- TEK HÜCRE. A100.
# Mel + f0 -> ses (44,1 kHz). NSF-HiFiGAN: perde f0'dan üretilen kaynaktan gelir (verilen frekansı çalar), tını mel'den.
# Veri: ses_nota/ses-modeli/vokoder-v1-paket/ · kod: ses_nota/ses-modeli/vokoder-kod.zip · çıktı: ses_nota/ses-modeli/runs/<tarih>-<AD>/
AD = 'vokoder-v1'       # koşu adı: runs/<tarih>-<AD>/ (aynı gün ikinci koşu -2, -3 ...; eski koşunun üstüne yazılmaz)
ADIM = 150000          # eğitim adımı (parti 16 kesit x 0,37 sn)
DEVAM = None          # oturum kesilirse: 'son' (bu AD'li en son koşunun son kayıt noktasından sürer)
KAPAT = True          # bitince Drive'ı kaydedip çalışma zamanını kapat (kota harcamasın)
A100_SART = True      # A100 yoksa eğitime başlamadan dur · False: yine de koş
AYAR = 'orta'         # üreteç boyu (model.AYARLAR): orta 3,7 M · hafif 1,6 M (daha hızlı, biraz daha düşük kalite)
PARTI = 16
BF16 = False          # True: bf16 otomatik karma (daha hızlı; GAN'da kararlılığı ilk koşuda denenmedi)
PAKET = 'vokoder-v1-paket'
# ----------------------------------------------------------------------------------------------------
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU yok'
print('GPU:', GPU)
if A100_SART and 'A100' not in GPU:
    raise SystemExit(f'A100 değil ({GPU}): Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU seç, yeniden '
                     'çalıştır. A100 olmadan koşmak istersen A100_SART = False yap.')
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil, zipfile
DRIVE = '/content/drive/MyDrive/ses_nota'
SM = f'{DRIVE}/ses-modeli'
KOD = '/content/vokoder-kod'
shutil.rmtree(KOD, ignore_errors=True)
zipfile.ZipFile(f'{SM}/vokoder-kod.zip').extractall(KOD)
sys.path.insert(0, KOD)
import vokoder_colab as vc
_kapat = KAPAT
A = None
try:
    A = vc.ayarlar(AD, ADIM, DEVAM, SM, GPU, KOD, AYAR=AYAR, PARTI=PARTI, BF16=BF16, PAKET=PAKET)
    vc.veri(A)            # Drive'dan /content'e (boyut denetimiyle) ve shard'ların açılması
    vc.hazirlik(A)        # koşu klasörü; DEVAM ise son kayıt noktası
    vc.egit(A)            # ADIM'a kadar; 10 dk'da bir son.pt, doğrulamada en iyi olunca en_iyi.pt (Drive'a)
    vc.olc(A)             # ölçüt pasajları (gerçek mel'den), girdi f0'ına perde sadakati, parçalı üretimde dikiş
except KeyboardInterrupt:  # elle durduruldu: çalışma zamanı açık kalsın
    _kapat = False
    raise
finally:
    if A is not None:
        vc.ozet(A)
    print('BİTTİ.')
    if _kapat:
        print('Drive kaydediliyor, çalışma zamanı kapatılıyor (KAPAT = True)...')
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()
    else:
        print('Çalışma zamanı → Çalışma zamanının bağlantısını kes ve sil (A100 boşta da kota harcar).')
